# TE-292 — Aula 3: juntando dados de manutenção

## Etapa B — Uso operacional e integração por ativo

**Tempo previsto:** 55 minutos

### O que aprenderemos

- resumir medições feitas ao longo do tempo para cada ativo;
- juntar tabelas que representam níveis de detalhe diferentes;
- verificar se a junção encontrou os ativos esperados;
- construir a tabela que será usada na demonstração da etapa C.

A sequência de medições ao longo do tempo é chamada de **série temporal**. Esta
etapa utiliza o arquivo produzido em A; portanto, execute
`A_maint_part_cons.ipynb` primeiro.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
%matplotlib inline

DATA_DIR = Path("dados_manutencao")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

usage_path = DATA_DIR / "usage_data.csv"
consumption_path = OUTPUT_DIR / "maintenance_part_consumption_by_reason.csv"
failures_path = DATA_DIR / "maintenance_failures.csv"

if not consumption_path.exists():
    raise FileNotFoundError(
        "A saída da etapa A não foi encontrada. Execute A_maint_part_cons.ipynb."
    )

usage = pd.read_csv(usage_path)
consumption_by_reason = pd.read_csv(consumption_path)
failures = pd.read_csv(failures_path)

print("Dimensões das entradas")
print(f"  uso:       {usage.shape}")
print(f"  consumo:   {consumption_by_reason.shape}")
print(f"  falhas:    {failures.shape}")

Dimensões das entradas
  uso:       (22828, 3)
  consumo:   (1913, 15)
  falhas:    (1805, 2)


## 1. Das medições para uma linha por ativo

Em `usage_data.csv`, cada ativo possui várias medições ao longo do tempo. Vamos
resumi-las para obter uma linha por ativo. Esse resumo é chamado de
**agregação**.

Calcularemos:

- `Time_min`: primeiro instante observado;
- `Use_min`: menor leitura de uso;
- `Use_max`: maior leitura de uso;
- `nb_km = Use_max - Use_min`: uso acumulado durante o período observado.

> Pergunta: por que a média de `Use` não representa o uso acumulado?

In [5]:
print(f"Eventos de uso: {len(usage):,}")
print(f"Ativos distintos: {usage['Asset'].nunique():,}")
print(f"Faltantes por coluna:\n{usage.isna().sum().to_string()}")

usage_by_asset = (
    usage
    .groupby("Asset", as_index=False)
    .agg(
        Time_min=("Time", "min"),
        Use_min=("Use", "min"),
        Use_max=("Use", "max"),
    )
)
usage_by_asset["nb_km"] = (
    usage_by_asset["Use_max"] - usage_by_asset["Use_min"]
).round(2)
usage_by_asset = usage_by_asset.drop(columns="Use_max")

assert usage_by_asset["Asset"].is_unique
assert (usage_by_asset["nb_km"] >= 0).all()

usage_output = OUTPUT_DIR / "usage_data_by_asset.csv"
usage_by_asset.to_csv(usage_output, index=False)

print(f"\nTabela agregada: {usage_by_asset.shape}")
print(f"Salva em: {usage_output}")
usage_by_asset.head()

Eventos de uso: 22,828
Ativos distintos: 1,894
Faltantes por coluna:
Asset    0
Time     0
Use      0

Tabela agregada: (1894, 4)
Salva em: output/usage_data_by_asset.csv


,Asset,Time_min,Use_min,nb_km
0,A000204,46,31449.650982,1763.11
1,A000270,435,26378.556339,658.21
2,A000463,30,30451.537250,1442.47
3,A000495,6,30851.249561,1689.00
4,A001201,395,26136.394866,1051.51


## 2. Juntando as tabelas

Juntar tabelas por uma coluna comum é uma operação chamada **join**. Nossa tabela
de referência será a de uso, pois queremos manter todos os ativos observados em
operação. Para isso usaremos um **left join**, que conserva todas as linhas da
tabela colocada à esquerda.

Antes da junção, verificamos se cada `Asset` aparece apenas uma vez em cada
tabela resumida. Essa relação esperada — uma linha de um lado para uma linha do
outro — é chamada de **cardinalidade um para um**.

No código, `validate="one_to_one"` interrompe a execução se essa regra não for
atendida, evitando que uma junção incorreta multiplique linhas silenciosamente.

In [7]:
for name, table in {
    "uso resumido": usage_by_asset,
    "consumo por motivo": consumption_by_reason,
    "situação de falha": failures,
}.items():
    duplicated = table["Asset"].duplicated().sum()
    print(f"{name:20s}: {len(table):4d} linhas; "
          f"{duplicated} ativos repetidos")
    assert duplicated == 0

integrated = usage_by_asset.merge(
    consumption_by_reason,
    on="Asset",
    how="left",
    validate="one_to_one",
    indicator="consumption_match",
)

print("\nAtivos encontrados na tabela de consumo")
print(integrated["consumption_match"].value_counts().to_string())

integrated = integrated.merge(
    failures,
    on="Asset",
    how="left",
    validate="one_to_one",
    indicator="failure_match",
)

print("\nAtivos encontrados na tabela de falhas")
print(integrated["failure_match"].value_counts().to_string())

uso resumido        : 1894 linhas; 0 ativos repetidos
consumo por motivo  : 1913 linhas; 0 ativos repetidos
situação de falha   : 1805 linhas; 0 ativos repetidos

Ativos encontrados na tabela de consumo
consumption_match
both          1893
left_only        1
right_only       0

Ativos encontrados na tabela de falhas
failure_match
both          1805
left_only       89
right_only       0


## 3. Verificando o resultado da junção

Um campo vazio após a junção não significa necessariamente erro:

- um motivo de consumo vazio pode indicar que não há evento registrado;
- `failure_bin` vazio indica que não conhecemos a situação de falha daquele
  ativo.

O nome técnico para um campo vazio é **valor faltante**. Manteremos esses campos
para não inventar informação. Na etapa C, o algoritmo LightGBM consegue trabalhar
com atributos faltantes, mas os ativos sem resposta conhecida serão separados.

A resposta que queremos prever é chamada de **alvo**; neste caso, o alvo é
`failure_bin`.

In [8]:
asset_modelization_table = (
    integrated
    .drop(columns=["consumption_match", "failure_match"])
)

missing_summary = pd.DataFrame({
    "faltantes": asset_modelization_table.isna().sum(),
    "percentual": asset_modelization_table.isna().mean(),
}).sort_values("percentual", ascending=False)

print(f"Tabela integrada: {asset_modelization_table.shape}")
print(f"Ativos únicos: {asset_modelization_table['Asset'].is_unique}")
print("\nColunas com maior percentual de faltantes")
display(missing_summary.head(10).style.format({"percentual": "{:.1%}"}))

model_output = OUTPUT_DIR / "asset_modelization_table.csv"
asset_modelization_table.to_csv(model_output, index=False)

print(f"Salva em: {model_output}")
asset_modelization_table.head()

Tabela integrada: (1894, 19)
Ativos únicos: True

Colunas com maior percentual de faltantes


,faltantes,percentual
R396,1893,99.9%
R782,1893,99.9%
R064,1892,99.9%
R606,1850,97.7%
R575,1803,95.2%
R044,1740,91.9%
R119,1595,84.2%
R417,1461,77.1%
R783,1408,74.3%
R446,720,38.0%


Salva em: output/asset_modelization_table.csv


,Asset,Time_min,Use_min,nb_km,R044,R064,R119,R193,R364,R396,R417,R446,R565,R575,R606,R707,R782,R783,failure_bin
0,A000204,46,31449.650982,1763.11,NaN,NaN,0.857143,1.629630,1.000000,NaN,NaN,1.000,3.785714,NaN,NaN,2.229885,NaN,NaN,1.0
1,A000270,435,26378.556339,658.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN
2,A000463,30,30451.537250,1442.47,NaN,NaN,1.000000,2.086957,1.000000,NaN,NaN,1.000,NaN,NaN,NaN,3.318841,NaN,1.5,1.0
3,A000495,6,30851.249561,1689.00,NaN,NaN,0.800000,2.928571,1.444444,NaN,NaN,1.625,8.040816,NaN,NaN,2.164179,NaN,NaN,0.0
4,A001201,395,26136.394866,1051.51,NaN,NaN,NaN,1.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.900000,NaN,NaN,0.0


## Ponto de verificação da etapa B

```text
uso por evento ── resumo ───────┐
                                ├── junção ── tabela por ativo
consumo por motivo ─────────────┤
falha por ativo ────────────────┘
```

- **Arquivos criados:** `usage_data_by_asset.csv` e
  `asset_modelization_table.csv`;
- **Coluna de ligação:** `Asset`, com um único valor por linha;
- **O que cada linha representa ao final:** um ativo.

O **inner join** é outro tipo de junção: ele conserva somente os ativos que
aparecem nas duas tabelas. **Discussão:** quais ativos seriam excluídos se o
usássemos no lugar do `left join`?